# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [44]:
import os
import json
import chromadb
from dotenv import load_dotenv
from lib.tooling import tool
from lib.llm import LLM
from lib.messages import SystemMessage, UserMessage
from lib.evaluation import EvaluationReport
from lib.parsers import PydanticOutputParser
from tavily import TavilyClient
from datetime import datetime
from lib.agents import Agent

In [2]:
# Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [29]:
# Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:

@tool
def retrieve_game(query):
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query (str): a question about game industry. 
    
    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    # Instantiate ChromaDB interfaces
    chroma_client = chromadb.PersistentClient(path="./chromadb")
    collection = chroma_client.get_collection("udaplay")

    # Get top 3 documents based on the query
    result = list()
    responses = collection.query(
        query_texts=[query],
        n_results=3,
    )
    for r in responses["metadatas"][0]:
        result.append(r)

    return result
    

In [21]:
# Testing the tool
racing_res = retrieve_game("Racing game on Playstation")
print(racing_res)

[{'Genre': 'Racing', 'Publisher': 'Sony Computer Entertainment', 'YearOfRelease': 1997, 'Name': 'Gran Turismo', 'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.', 'Platform': 'PlayStation 1'}, {'Name': 'Gran Turismo 5', 'Genre': 'Racing', 'Publisher': 'Sony Computer Entertainment', 'Platform': 'PlayStation 3', 'YearOfRelease': 2010, 'Description': 'A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.'}, {'Description': "An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson.", 'Name': 'Grand Theft Auto: San Andreas', 'Platform': 'PlayStation 2', 'Publisher': 'Rockstar Games', 'YearOfRelease': 2004, 'Genre': 'Action-adventure'}]


#### Evaluate Retrieval Tool

In [28]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:

@tool
def evaluate_retrieval(question, retrieved_docs):
    """
    Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question.
    
    args: 
    - question (str): original question from user
    - retrieved_docs (list): retrieved documents most similar to the user query in the Vector Database
    
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    # Instantiate LLM
    llm = LLM(api_key=OPENAI_API_KEY)

    # Build prompt
    system_prompt = f"""
    Your task is to evaluate if the documents are enough to respond to the query.
    Give a detailed explanation, so it's possible to take an action to accept it or not.
    """
    user_prompt = f"""
    ### Question
    {question}

    ### Retrieved Documents
    {retrieved_docs}
    """
    messages = [SystemMessage(content=system_prompt), UserMessage(content=user_prompt)]

    # Invoke LLM and parse result
    response = llm.invoke(
        input=messages,
        response_format=EvaluationReport
    )
    parser = PydanticOutputParser(model_class=EvaluationReport)
    try:
        evaluation = parser.parse(response)
    except Exception as e:
        print(f"Debug: Structured parsing error: {e}")
        print(f"Debug: LLM response content: {response.content}")

    return dict(evaluation)

In [25]:
# Testing the tool
racing_eval_res = evaluate_retrieval(
    question="Racing game on Playstation",
    retrieved_docs=racing_res
)

In [26]:
racing_eval_res

{'useful': True,
 'description': "The retrieved documents provide relevant information about racing games available on PlayStation. Specifically, they include two notable racing titles: 'Gran Turismo' and 'Gran Turismo 5', both of which are highly regarded in the racing genre and are published by Sony Computer Entertainment. The documents detail the genre, publisher, year of release, and descriptions of these games, which are essential for understanding their significance in the context of racing games on PlayStation. Although one document pertains to 'Grand Theft Auto: San Andreas', which is not a racing game, the presence of the two Gran Turismo titles sufficiently addresses the query about racing games on PlayStation. Therefore, the documents are deemed useful for responding to the query."}

#### Game Web Search Tool

In [40]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:

@tool
def game_web_search(question):
    """
    Web search: Finds most relevant results from the internet
    
    args:
    - question (str): a question about game industry.
    """
    # Instantiate the web-search client
    client = TavilyClient(api_key=TAVILY_API_KEY)

    # Get top 3 from search result
    result = client.search(query=question, include_answer=True)
    formatted_results = {
        "answer": result.get("answer", ""),
        "results": result["results"][:3] if result.get("results") else [],
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": question
        }
    }

    return formatted_results

In [41]:
# Testing the tool
racing_web_res = game_web_search("Racing game on Playstation")

In [42]:
racing_web_res

{'answer': 'Gran Turismo, Need for Speed Unbound, and F1 25 are top racing games on PlayStation. Gran Turismo is highly regarded for its realism. Need for Speed Unbound offers street racing with customization.',
 'results': [{'url': 'https://www.playstation.com/en-us/editorial/great-racing-games-on-ps4',
   'title': 'The best racing games on PS4 & PS5 - Guides & Editorial | PlayStation',
   'content': 'The best racing games on PS4. Great racing games on PS4 and PS5 - artwork. Rev your engine for our rundown of great racing games available for your inner speed demon. ## Kart racing games. Sonic Racing: CrossWorlds screenshot showing Shadow racing. ### Sonic Racing: CrossWorlds. Drift across dimensions as\xa0Sonic and friends (and enemies) race on land and sea, in the air and even through space. Sonic Racing: CrossWorlds - Launch Trailer | PS5 & PS4 Games. Sonic Racing: CrossWorlds - Announce Trailer | PS5 & PS4 Games. Crash Team Racing: Nitro-Fueled screenshot showing Crash racing again

### Agent

In [68]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
system_prompt = f"""
# Role & Task
You are an expert researcher of the video game industry.
Only answer questions about the video game industry, otherwise say you can't answer them.
You must answer the user's query with only the knowledge provided by the tools you call.
You are provided two tools to obtain knowledge: semantic-search tool on your own DB and the web search.
When acquiring knowledge, first search the essential knowledge from the DB.
The knowledge from the DB must be evaluated by the evaluate_retrieval tool.
When the knowledge acquired from the DB is not useful, search the web.
You can also run the evaluation tool with the knowledge from the web.
If there is no useful knowledge found at all, just say you don't know the answer and apologize.
You must always cite your sources, whether the web (with the URL) or the DB, and explain any discrepancies found.
"""

gaming_agent = Agent(
    model_name = "gpt-4o-mini",
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    instructions=system_prompt,
    api_key=OPENAI_API_KEY
)

In [51]:
# Helper tool to print messages
def print_messages(messages):
    for m in messages:
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

#### Test Query 1

In [69]:
# Test Query 1
pokemon_res = gaming_agent.invoke(
    query="When Pokémon Gold and Silver was released?",
    session_id="udaplay"
)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


In [70]:
print_messages(pokemon_res.get_final_state()["messages"])

 -> (role = system, content = 
# Role & Task
You are an expert researcher of the video game industry.
Only answer questions about the video game industry, otherwise say you can't answer them.
You must answer the user's query with only the knowledge provided by the tools you call.
You are provided two tools to obtain knowledge: semantic-search tool on your own DB and the web search.
When acquiring knowledge, first search the essential knowledge from the DB.
The knowledge from the DB must be evaluated by the evaluate_retrieval tool.
When the knowledge acquired from the DB is not useful, search the web.
You can also run the evaluation tool with the knowledge from the web.
If there is no useful knowledge found at all, just say you don't know the answer and apologize.
You must always cite your sources, whether the web (with the URL) or the DB, and explain any discrepancies found.
, tool_calls = None)
 -> (role = user, content = When Pokémon Gold and Silver was released?, tool_calls = None)


In [79]:
# Final Answer
pokemon_res.get_final_state()["messages"][-1].content

'Pokémon Gold and Silver was released in 1999 for the Game Boy Color. These games are part of the second generation of Pokémon, introducing new regions and gameplay mechanics (source: DB).'

#### Test Query 2

In [71]:
# Test Query 2
mario_res = gaming_agent.invoke(
    query="Which one was the first 3D platformer Mario game?",
    session_id="udaplay"
)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


In [72]:
print_messages(mario_res.get_final_state()["messages"])

 -> (role = system, content = 
# Role & Task
You are an expert researcher of the video game industry.
Only answer questions about the video game industry, otherwise say you can't answer them.
You must answer the user's query with only the knowledge provided by the tools you call.
You are provided two tools to obtain knowledge: semantic-search tool on your own DB and the web search.
When acquiring knowledge, first search the essential knowledge from the DB.
The knowledge from the DB must be evaluated by the evaluate_retrieval tool.
When the knowledge acquired from the DB is not useful, search the web.
You can also run the evaluation tool with the knowledge from the web.
If there is no useful knowledge found at all, just say you don't know the answer and apologize.
You must always cite your sources, whether the web (with the URL) or the DB, and explain any discrepancies found.
, tool_calls = None)
 -> (role = user, content = When Pokémon Gold and Silver was released?, tool_calls = None)


In [80]:
# Final Answer
mario_res.get_final_state()["messages"][-1].content

'The first 3D platformer Mario game is **Super Mario 64**, which was released in 1996 for the Nintendo 64. This game is acclaimed for setting new standards in the platformer genre (source: DB).'

#### Test Query 3

In [81]:
# Test Query 3
mortal_kombat_res = gaming_agent.invoke(
    query="Was Mortal Kombat X released for Playstation 5?",
    session_id="udaplay"
)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


In [83]:
print_messages(mortal_kombat_res.get_final_state()["messages"])

 -> (role = system, content = 
# Role & Task
You are an expert researcher of the video game industry.
Only answer questions about the video game industry, otherwise say you can't answer them.
You must answer the user's query with only the knowledge provided by the tools you call.
You are provided two tools to obtain knowledge: semantic-search tool on your own DB and the web search.
When acquiring knowledge, first search the essential knowledge from the DB.
The knowledge from the DB must be evaluated by the evaluate_retrieval tool.
When the knowledge acquired from the DB is not useful, search the web.
You can also run the evaluation tool with the knowledge from the web.
If there is no useful knowledge found at all, just say you don't know the answer and apologize.
You must always cite your sources, whether the web (with the URL) or the DB, and explain any discrepancies found.
, tool_calls = None)
 -> (role = user, content = When Pokémon Gold and Silver was released?, tool_calls = None)


In [84]:
# Final Answer
mortal_kombat_res.get_final_state()["messages"][-1].content

'Mortal Kombat X was not released for PlayStation 5; it was originally released for PlayStation 4. Although there may be gameplay reviews and discussions about the game on PS5, it is not a native release for that platform (source: [Wikipedia](https://en.wikipedia.org/wiki/Mortal_Kombat_X)).'

#### Test Query 4: Remembering Previous Queries

In [85]:
# Test Query 4
remembering_res = gaming_agent.invoke(
    query="Summarize all your answers from this session!",
    session_id="udaplay"
)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


In [87]:
# Final Answer
print(remembering_res.get_final_state()["messages"][-1].content)

1. **Pokémon Gold and Silver Release**: These games were released in 1999 for the Game Boy Color, introducing new regions and gameplay mechanics.

2. **First 3D Platformer Mario Game**: The first 3D platformer Mario game is **Super Mario 64**, released in 1996 for the Nintendo 64, which set new standards for the platformer genre.

3. **Mortal Kombat X for PlayStation 5**: Mortal Kombat X was not released for PlayStation 5; it was originally released for PlayStation 4. Although it may be playable on PS5, it is not a native release for that console. 

If you have any more questions or need further information, feel free to ask!


In [89]:
# Total Runs in a Session
udaplay_runs = gaming_agent.get_session_runs("udaplay")
print(udaplay_runs)

[Run('14f1f3ee-5041-431b-bd53-f07759e3290b'), Run('59b41cdc-171a-4652-bab7-f5e5d2717b82'), Run('c75ff8f9-7771-4789-b1a6-cc801768cd9c'), Run('c080e09e-6d33-4fdc-ba0b-2184a25c13c2')]


---
**Remarks**/n
As demonstrated above, the `gaming_agent` is able to use all three of the tools provided. It also shows its ability to fallback on the web-search whenever the `evaluate_retrieval` returns a negative result on the retrieved documents' usefulness.

Also observed above, the agent is employing its internal state-machine and short-term-memory as declared and applied in the `/lib/agents.py`.